In [ ]:
from sympy import *
from Rate_laws import *
import numpy as np
import scipy.integrate as sp
import matplotlib.pyplot as plt
import random as rd

In [ ]:
# this function is used to select parameters randomly

def select_par():
    list_par = list(np.random.uniform(low=0, high=10, size=18))
    return list_par

In [ ]:

compounds = ["Acetyl_CoA", "Malonyl_CoA", "Fatty_Acyl", "TAG"]

n_compounds = len(compounds)



par = {
    "k1": 1.0,
    "k2": 1.0,
    "k3": 1.0,
    "alpha": 1.0,
    "beta": 1.0,
    "gamma": 1.0,
    "V1": 1.0,
    "V2": 1.0,
    "V3": 1.0,
    "V4": 1.0,
    "V5": 1.0,
    "Km1": 1.0,
    "Km2": 1.0,
    "Km3": 1.0,
    "Km4": 1.0,
    "Km5": 1.0,
    "q1": 1.0,
    "q4": 1.0
}

initial = {"y0": np.random.random(n_compounds)}

In [ ]:
# This part of the code solves the system and simulate the steady when the initial conditions are choosen randomly

##################################### rate equations for individual processes ######################################


def get_rates(y, par):
    Acetyl_CoA, Malonyl_CoA, Fatty_Acyl, TAG = y
    return {
        "v_in_1":
        constant_influx(par["k1"]),
        "v_in_2":
        constant_influx(par["k2"]),

        # "v_in_3": constant_influx(par["Kin3"]),
        "v_in_3":
        constant_influx(par["k3"]),
        "r1":
        non_competitive_inhibition(par["V1"], par["Km1"], par["q1"],
                                   Acetyl_CoA, Fatty_Acyl),
        "r2":
        michaelis_meten(par["V2"], par["Km2"], Malonyl_CoA),
        "r3":
        michaelis_meten(par["V3"], par["Km3"], Fatty_Acyl),
        "r4":
        non_competitive_inhibition(par["V4"], par["Km4"], par["q4"],
                                   Fatty_Acyl, Malonyl_CoA),
        "r5":
        michaelis_meten(par["V5"], par["Km5"], TAG),
        "v_out1":
        mass_action_law(par["alpha"], Acetyl_CoA),
        "v_out2":
        mass_action_law(par["beta"], Fatty_Acyl),
        "v_out3":
        mass_action_law(par["gamma"], TAG),
    }


############################################## Differential equations describing the process #####################################################################


def model(time, y, par):
    rates = get_rates(y=y, par=par)
    d_Acetyl_CoA_dt = (rates["v_in_1"] - rates["r1"] + rates["r4"] -
                       rates["v_out1"])
    d_Malony_CoA = rates["r1"] - rates["r2"]

    d_Fatty_Acyl = (rates["v_in_2"] + rates["r2"] - rates["r3"] - rates["r4"] +
                    rates["r5"] - rates["v_out2"])

    d_TAG_dt = rates["v_in_3"] + rates["r3"] - rates["r5"] - rates["v_out3"]

    return np.array([d_Acetyl_CoA_dt, d_Malony_CoA, d_Fatty_Acyl, d_TAG_dt])


######################### Initial conditions + time points ################################################################

y0 = initial["y0"]
time = np.linspace(0, 10, 100)
y = sp.odeint(func=model, y0=y0, t=time, args=(par, ), tfirst=True)

# Take a look at these results first, before you run the steady state simulation

########################## Plot the solutions #################################################################################

# plt.style.use('ggplot')
# fig, ax = plt.subplots()
# ax.plot(time, y)
# plt.xlabel("Time (a.u.) ")
# plt.ylabel("Concemtration (a.u.) ")
# plt.title("Concentrations over time ")
# ax.legend(compounds,
#           loc="upper left",
#           bbox_to_anchor=(1.1, 1),
#           borderaxespad=0)
# plt.tight_layout()
# plt.savefig("Concentrations.png", bbox_inches='tight')
# plt.show()


######################## Simulate the steady-state function as function of parameters#############################################
def simulate_steady_state(model, y0, par, dt=1e-3, error=1e-5, max_steps=1e8):
    r = sp.ode(model)
    r.set_integrator("lsoda", method="bdf")
    r.set_f_params(par)
    r.set_initial_value(y0, 0.0)

    y1 = y0.copy()
    nsteps = 0
    while r.successful and nsteps < max_steps:
        y2 = r.integrate(r.t + dt)
        if np.linalg.norm(y1 - y2, ord=2) < error:
            return y2
        else:
            y1 = y2
        nsteps += 1
    raise ValueError("Maximal simulation steps exceeded")


steady = list(simulate_steady_state(model, y0, par, dt=100))

In [ ]:
S1 = steady[0]
S2 = steady[1]
S4 = steady[3]

In [ ]:
# Random update of the list
#par = {key: value for key, value in zip(par.keys(), select_par())}

In [ ]:
#Compute the values of the coefficients 


def coefficient_P2(par):
   # par = {key: value for key, value in zip(par.keys(), select_par())}

    A2 = (-par["Km2"] * par["Km5"] * S2 * par["beta"] * par["q4"] -
          par["Km2"] * par["Km5"] * par["beta"] -
          par["Km2"] * S2 * S4 * par["beta"] * par["q4"] -
          par["Km2"] * S4 * par["beta"] - par["Km5"] *
          (S2**2) * par["beta"] * par["q4"] - par["Km5"] * S2 * par["beta"] -
          (S2**2) * S4 * par["beta"] * par["q4"] - S2 * S4 * par["beta"])

    B2 = (-par["Km2"] * par["Km3"] * par["Km5"] * S2 * par["beta"] * par["q4"]
          - par["Km2"] * par["Km3"] * par["Km5"] * par["beta"] -
          par["Km2"] * par["Km3"] * S4 * par["beta"] -
          par["Km2"] * par["Km4"] * par["Km5"] * S2 * par["beta"] * par["q4"] -
          par["Km2"] * par["Km4"] * par["Km5"] * par["beta"] -
          par["Km2"] * par["Km4"] * S2 * S4 * par["beta"] * par["q4"] -
          par["Km2"] * par["Km4"] * S4 * par["beta"] -
          par["Km2"] * par["Km5"] * S2 * par["V3"] * par["q4"] +
          par["Km2"] * par["Km5"] * S2 * par["k2"] * par["q4"] -
          2 * par["Km2"] * par["Km5"] * par["V3"] +
          par["Km2"] * par["Km5"] * par["k2"] -
          par["Km2"] * S2 * S4 * par["V3"] * par["q4"] +
          par["Km2"] * S2 * S4 * par["V5"] * par["q4"] +
          par["Km2"] * S2 * S4 * par["k2"] * par["q4"] -
          2 * par["Km2"] * S4 * par["V3"] + par["Km2"] * S4 * par["V5"] +
          par["Km2"] * S4 * par["k2"] - par["Km3"] * par["Km5"] *
          (S2**2) * par["beta"] * par["q4"] -
          par["Km3"] * par["Km5"] * S2 * par["beta"] - par["Km3"] *
          (S2**2) * S4 * par["beta"] * par["q4"] -
          par["Km3"] * S2 * S4 * par["beta"] - par["Km4"] * par["Km5"] *
          (S2**2) * par["beta"] * par["q4"] - par["Km5"] *
          (S2**2) * par["V3"] * par["q4"] -
          par["Km4"] * par["Km5"] * S2 * par["beta"] - par["Km4"] *
          (S2**2) * S4 * par["beta"] * par["q4"] -
          -par["Km4"] * S2 * S4 * par["beta"] + par["Km5"] *
          (S2**2) * par["V2"] * par["q4"] - par["Km5"] *
          (S2**2) * par["V3"] * par["q4"] + par["Km5"] *
          (S2**2) * par["k2"] * par["q4"] + par["Km5"] * S2 * par["V2"] -
          2 * par["Km5"] * S2 * par["V3"] + par["Km5"] * S2 * par["k2"] +
          (S2**2) * S4 * par["V2"] * par["q4"] -
          (S2**2) * S4 * par["V3"] * par["q4"] +
          (S2**2) * S4 * par["V5"] * par["q4"] +
          (S2**2) * S4 * par["k2"] * par["q4"] + S2 * S4 * par["V2"] -
          2 * S2 * S4 * par["V3"] + S2 * S4 * par["V5"] + S2 * S4 * par["k2"])

    C2 = (-par["Km2"] * par["Km3"] * par["Km4"] * par["Km5"] * S2 *
          par["beta"] * par["q4"] -
          par["Km2"] * par["Km3"] * par["Km4"] * par["Km5"] * par["beta"] -
          par["Km2"] * par["Km3"] * par["Km4"] * S2 * S4 * par["beta"] *
          par["q4"] - par["Km2"] * par["Km3"] * par["Km4"] * S4 * par["beta"] +
          par["Km2"] * par["Km3"] * par["Km5"] * S2 * par["k2"] * par["q4"] -
          par["Km2"] * par["Km3"] * par["Km5"] * par["V3"] +
          par["Km2"] * par["Km3"] * par["Km5"] * par["k2"] +
          par["Km2"] * par["Km3"] * S2 * S4 * par["V5"] * par["q4"] +
          par["Km2"] * par["Km3"] * S2 * S4 * par["k2"] * par["q4"] -
          par["Km2"] * par["Km3"] * S4 * par["V3"] +
          par["Km2"] * par["Km3"] * S4 * par["V5"] +
          par["Km2"] * par["Km3"] * S4 * par["k2"] -
          par["Km2"] * par["Km4"] * par["Km5"] * S2 * par["V3"] * par["q4"] +
          par["Km2"] * par["Km4"] * par["Km5"] * S2 * par["k2"] * par["q4"] -
          par["Km2"] * par["Km4"] * par["Km5"] * par["V3"] +
          par["Km2"] * par["Km4"] * par["Km5"] * par["k2"] -
          par["Km2"] * par["Km4"] * S2 * S4 * par["V3"] * par["q4"] +
          par["Km2"] * par["Km4"] * S2 * S4 * par["V5"] * par["q4"] +
          par["Km2"] * par["Km4"] * S2 * S4 * par["k2"] * par["q4"] -
          par["Km2"] * par["Km4"] * S4 * par["V3"] +
          par["Km2"] * par["Km4"] * S4 * par["V5"] +
          par["Km2"] * par["Km4"] * S4 * par["k2"] -
          par["Km3"] * par["Km4"] * par["Km5"] *
          (S2**2) * par["beta"] * par["q4"] -
          par["Km3"] * par["Km4"] * par["Km5"] * S2 * par["beta"] -
          par["Km3"] * par["Km4"] * (S2**2) * S4 * par["beta"] * par["q4"] -
          par["Km3"] * par["Km4"] * S2 * S4 * par["beta"] +
          par["Km3"] * par["Km5"] * (S2**2) * par["V2"] * par["q4"] +
          par["Km3"] * par["Km5"] * (S2**2) * par["k2"] * par["q4"] +
          par["Km3"] * par["Km5"] * S2 * par["V2"] -
          par["Km3"] * par["Km5"] * S2 * par["V3"] +
          par["Km3"] * par["Km5"] * S2 * par["k2"] + par["Km3"] *
          (S2**2) * S4 * par["V2"] * par["q4"] + par["Km3"] *
          (S2**2) * S4 * par["V5"] * par["q4"] + par["Km3"] *
          (S2**2) * S4 * par["k2"] * par["q4"] +
          par["Km3"] * S2 * S4 * par["V2"] - par["Km3"] * S2 * S4 * par["V3"] +
          par["Km3"] * S2 * S4 * par["V5"] + par["Km3"] * S2 * S4 * par["k2"] +
          par["Km4"] * par["Km5"] * (S2**2) * par["V2"] * par["q4"] -
          par["Km4"] * par["Km5"] * (S2**2) * par["V3"] * par["q4"] +
          par["Km4"] * par["Km5"] * (S2**2) * par["k2"] * par["q4"] +
          par["Km4"] * par["Km5"] * S2 * par["V2"] -
          par["Km4"] * par["Km5"] * S2 * par["V3"] +
          par["Km4"] * par["Km5"] * S2 * par["k2"] + par["Km4"] *
          (S2**2) * S4 * par["V2"] * par["q4"] - par["Km4"] *
          (S2**2) * S4 * par["V3"] * par["q4"] + par["Km4"] *
          (S2**2) * S4 * par["V5"] * par["q4"] + par["Km4"] *
          (S2**2) * S4 * par["k2"] * par["q4"] +
          par["Km4"] * S2 * S4 * par["V2"] - par["Km4"] * S2 * S4 * par["V3"] +
          par["Km4"] * S2 * S4 * par["V5"] + par["Km4"] * S2 * S4 * par["k2"])

    D2 = (par["Km2"] * par["Km3"] * par["Km4"] * par["Km5"] * S2 * par["k2"] *
          par["q4"] +
          par["Km2"] * par["Km3"] * par["Km4"] * par["Km5"] * par["k2"] +
          par["Km2"] * par["Km3"] * par["Km4"] * S2 * S4 * par["V5"] *
          par["q4"] + par["Km2"] * par["Km3"] * par["Km4"] * S2 * S4 *
          par["k2"] * par["q4"] +
          par["Km2"] * par["Km3"] * par["Km4"] * S4 * par["V5"] +
          par["Km2"] * par["Km3"] * par["Km4"] * S4 * par["k2"] +
          par["Km3"] * par["Km4"] * par["Km5"] *
          (S2**2) * par["V2"] * par["q4"] +
          par["Km3"] * par["Km4"] * par["Km5"] *
          (S2**2) * par["k2"] * par["q4"] +
          par["Km3"] * par["Km4"] * par["Km5"] * S2 * par["V2"] +
          par["Km3"] * par["Km4"] * par["Km5"] * S2 * par["k2"] +
          par["Km3"] * par["Km4"] * (S2**2) * S4 * par["V2"] * par["q4"] +
          par["Km3"] * par["Km4"] * (S2**2) * S4 * par["V5"] * par["q4"] +
          par["Km3"] * par["Km4"] * (S2**2) * S4 * par["k2"] * par["q4"] +
          par["Km3"] * par["Km4"] * S2 * S4 * par["V2"] +
          par["Km3"] * par["Km4"] * S2 * S4 * par["V5"] +
          +par["Km3"] * par["Km4"] * S2 * S4 * par["k2"])
    
    return [A2, B2, C2, D2, par]
coefficient_P2(par)

In [ ]:

def condition(par):
    par = {key: value for key, value in zip(par.keys(), select_par())}

    
    A2 = coefficient_P2(par)[0]
    B2 = coefficient_P2(par)[1]
    C2 = coefficient_P2(par)[2]
    D2 = coefficient_P2(par)[3]
    condition1 = B2
    condition2 = C2
    condition3 = B2**2 - 3 * A2 * C2
    condition4 = D2 - (B2*C2/(3*A2)) + ((2*C2*np.sqrt(B2**2 - 3*A2*C2))/(9*A2)) + \
        (2*B2**3/(27*A2**2)) - (((2*B2**2)*np.sqrt(B2**2 - 3*A2*C2))/(27*A2**2))
    condition5 = D2 - (B2*C2/(3*A2)) - ((2*C2*np.sqrt(B2**2 - 3*A2*C2))/(9*A2)) + \
        (2*B2**3/(27*A2**2)) + (((2*B2**2)*np.sqrt(B2**2 - 3*A2*C2))/(27*A2**2))
    
    return [condition1, condition2, condition3, condition4, condition5, par]
    

condition(par)

In [ ]:
X = coefficient_P2(par)
A = condition(par)

condition1 = A[0]
condition2 = A[1]
condition3 = A[2]
condition4 = A[3]
condition5 = A[4]
B = A[5]

step = 0
while step < 1000000:
    X = coefficient_P2(par)
    A = condition(par)

    condition1 = A[0]
    condition2 = A[1]
    condition3 = A[2]
    condition4 = A[3]
    condition5 = A[4]
    B = A[5]
    X = X
    A = A

    if condition1 > 0 and condition2 < 0 and condition3 > 0 and condition4 <0 and condition5>0:
        print("Got it" , B)
        break
    else:
        print("Not Good please try again" )
        X = X
        A = A
        

    step = step + 1
    
print("finish")
